In [ ]:
%pip install langchain langchain-core langchain-mcp-adapters fastmcp openai

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")


In [ ]:
import asyncio
from mcp.client.session import ClientSession
from mcp.client.stdio import stdio_client, StdioServerParameters
from openai import OpenAI

client = OpenAI()  # uses OPENAI_API_KEY from env

async def main():
    server = StdioServerParameters(
        command="python",
        args=["math_server.py"]   #make sure filename is correct
    )
    # Connect to MCP server via stdio
    async with stdio_client(server) as (read, write):

        async with ClientSession(read, write) as session:
            # Load tools from server
            await session.initialize()

            user_question = "What is 12 + 30?"

            # Ask OpenAI model
            response = client.responses.create(
                model="gpt-4.1-mini",
                input=user_question,
                tools=session.tools
            )

            # If model wants to call a tool
            for item in response.output:
                if item["type"] == "tool_call":
                    tool_name = item["name"]
                    args = item["arguments"]

                    # Call MCP tool
                    result = await session.call_tool(tool_name, args)

                    # Send tool result back to model
                    final_response = client.responses.create(
                        model="gpt-4.1-mini",
                        input=[
                            {"role": "user", "content": user_question},
                            item,
                            {
                                "role": "tool",
                                "tool_name": tool_name,
                                "content": str(result.content)
                            }
                        ]
                    )

                    print(final_response.output_text)

await main()
